![imagen](./img/python.jpg)

# Extra: archivos Parquet

En el notebook de lectura/escritura has visto CSV, Excel, JSON, TXT, ZIP y pickle. Todos ellos son formatos muy habituales, pero cuando los datos empiezan a crecer (millones de filas, decenas de columnas) aparece otro protagonista en el mundo del dato: **Parquet**.

En este notebook extra verás qué es, por qué se usa tanto en entornos profesionales y cómo trabajar con él desde `pandas`.

1. [¿Qué es Parquet?](#1.-¿Qué-es-Parquet?)
2. [Instalación](#2.-Instalación)
3. [Escritura y lectura](#3.-Escritura-y-lectura)
4. [Conserva los tipos de datos](#4.-Conserva-los-tipos-de-datos)
5. [Tamaño y velocidad](#5.-Tamaño-y-velocidad)
6. [Leer solo lo que necesitas](#6.-Leer-solo-lo-que-necesitas)
7. [Particionado](#7.-Particionado)
8. [¿Cuándo usar cada formato?](#8.-¿Cuándo-usar-cada-formato?)

## 1. ¿Qué es Parquet?
**Apache Parquet es un formato de archivo binario, columnar y comprimido**, pensado para almacenar datos tabulares de forma eficiente. Es open source y es el estándar *de facto* en *data lakes*, Spark, BigQuery, Athena, Snowflake, Databricks...

Vamos por partes:

1. **Binario**: a diferencia del CSV o el JSON, no es texto plano. Si lo abres con el bloc de notas verás caracteres sin sentido. Necesitas una librería para leerlo.
2. **Columnar**: los datos se guardan **columna a columna**, no fila a fila.
3. **Comprimido**: como los valores de una misma columna se parecen mucho entre sí (mismo tipo, valores repetidos...), se comprimen muy bien.
4. **Con esquema**: el propio archivo guarda los nombres de columnas y **sus tipos de datos**. No hay que adivinar nada al leerlo.

**¿Qué significa eso de columnar?** Imagina esta tabla:

| season  | localTeam  | localGoals |
|---------|------------|------------|
| 2005-06 | Alaves     | 0          |
| 2005-06 | Valencia   | 1          |
| 2005-06 | Cadiz      | 1          |

Un **CSV (orientado a filas)** la guarda así:
```
2005-06,Alaves,0
2005-06,Valencia,1
2005-06,Cadiz,1
```

Un **Parquet (orientado a columnas)** la guarda, conceptualmente, así:
```
season:     2005-06, 2005-06, 2005-06
localTeam:  Alaves, Valencia, Cadiz
localGoals: 0, 1, 1
```

Esto tiene dos ventajas enormes:
* Si solo necesitas la columna `localGoals`, **puedes leer únicamente esa columna** sin tocar el resto del archivo.
* Fíjate en `season`: el mismo valor repetido tres veces. Guardarlo como "2005-06 x3" ocupa mucho menos. **Por eso Parquet comprime tanto**.

[Documentación oficial de Apache Parquet](https://parquet.apache.org/docs/)

## 2. Instalación
`pandas` no lee Parquet por sí solo: necesita un motor. Los dos disponibles son `pyarrow` (el recomendado y el que usa por defecto) y `fastparquet`. Con instalar uno de ellos es suficiente.

In [ ]:
# %pip install pyarrow

In [ ]:
import os
import pandas as pd

## 3. Escritura y lectura
Partimos de nuestro ya conocido CSV de la liga.

In [ ]:
df = pd.read_csv('data/laliga.csv', index_col = 'Unnamed: 0')
df.head()

**Escritura de Parquet**

Igual que teníamos `to_csv()` o `to_excel()`, ahora tenemos `to_parquet()`. Tienes [el enlace a la documentación para ver más detalle](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.to_parquet.html).

In [ ]:
df.to_parquet('data/laliga.parquet')

**Lectura de Parquet**

Y para leer, `read_parquet()`. [Aquí tienes la documentación](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_parquet.html).

Fíjate en que, a diferencia del CSV, **no hace falta indicar separador, cabecera, encoding ni columna índice**: todo eso va dentro del archivo. El índice se ha guardado y se recupera tal cual.

In [ ]:
df_parquet = pd.read_parquet('data/laliga.parquet')
df_parquet.head()

In [ ]:
# ¿Son exactamente iguales?
df.equals(df_parquet)

## 4. Conserva los tipos de datos
Este es, probablemente, **el motivo más práctico para usar Parquet en el día a día**.

Cuando trabajamos con un CSV, todo es texto. Si limpiamos los datos y convertimos una columna a fecha, al guardar en CSV y volver a leer... **la fecha vuelve a ser un string**. Y tenemos que repetir la conversión cada vez que leemos el archivo.

Vamos a limpiar un poco nuestro `DataFrame`:

In [ ]:
df['date'] = pd.to_datetime(df['date'], format = '%d/%m/%Y')
df['season'] = df['season'].astype('category')
df['division'] = df['division'].astype('int8')

df.dtypes

Lo guardamos en ambos formatos y volvemos a leer.

In [ ]:
df.to_csv('data/laliga_tipos.csv')
df.to_parquet('data/laliga_tipos.parquet')

df_csv = pd.read_csv('data/laliga_tipos.csv', index_col = 0)
df_pq = pd.read_parquet('data/laliga_tipos.parquet')

pd.DataFrame({'original': df.dtypes,
              'csv': df_csv.dtypes,
              'parquet': df_pq.dtypes})

**El CSV ha perdido la fecha, la categoría y el `int8`. El Parquet los conserva todos.** Limpias una vez, guardas en Parquet, y el resto del equipo (o tú mismo mañana) lee los datos ya listos.

## 5. Tamaño y velocidad
Comparamos cuánto ocupa cada archivo en disco con `os.path.getsize()`, que devuelve el tamaño en bytes.

In [ ]:
for archivo in ['data/laliga_tipos.csv', 'data/laliga_tipos.parquet']:
    print(archivo, round(os.path.getsize(archivo) / 1024, 1), 'KB')

**Compresión**

Por defecto, `to_parquet()` comprime con `snappy` (rápido, compresión moderada). Podemos elegir otros algoritmos con el argumento `compression`:
* `'snappy'`: por defecto. Buen equilibrio.
* `'gzip'`: comprime más, pero es más lento.
* `'zstd'`: comprime mucho y es rápido. Muy recomendable.
* `None`: sin compresión.

In [ ]:
for compresion in ['snappy', 'gzip', 'zstd', None]:
    ruta = f'data/laliga_{compresion}.parquet'
    df.to_parquet(ruta, compression = compresion)
    print(compresion, round(os.path.getsize(ruta) / 1024, 1), 'KB')

**Velocidad de lectura**

Con el comando mágico `%timeit` de Jupyter medimos cuánto tarda en ejecutarse una línea (la repite varias veces y hace la media).

In [ ]:
%timeit pd.read_csv('data/laliga_tipos.csv')

In [ ]:
%timeit pd.read_parquet('data/laliga_tipos.parquet')

Con un dataset de ~5.000 filas la diferencia es pequeña en valor absoluto. **Con millones de filas, la diferencia entre CSV y Parquet pasa de segundos a minutos**, tanto en tiempo de lectura como en espacio en disco.

## 6. Leer solo lo que necesitas
Aquí es donde brilla el formato columnar.

**Seleccionar columnas**: con `columns` solo se leen del disco las columnas indicadas. En un CSV, aunque uses `usecols`, hay que recorrer el archivo entero igualmente.

In [ ]:
pd.read_parquet('data/laliga_tipos.parquet', columns = ['localTeam', 'localGoals']).head()

**Filtrar filas**: con `filters` podemos indicar condiciones, en forma de lista de tuplas `(columna, operador, valor)`. Parquet guarda estadísticas (mínimo, máximo...) por bloques de filas, por lo que puede **saltarse los bloques que no cumplen la condición** sin leerlos.

Operadores disponibles: `'=='`, `'!='`, `'<'`, `'>'`, `'<='`, `'>='`, `'in'`, `'not in'`.

In [ ]:
pd.read_parquet('data/laliga_tipos.parquet',
                columns = ['season', 'localTeam', 'visitorTeam', 'localGoals'],
                filters = [('localGoals', '>=', 6)])

Varias condiciones dentro de la misma lista se combinan con un AND.

In [ ]:
pd.read_parquet('data/laliga_tipos.parquet',
                filters = [('localTeam', '==', 'Real Madrid'),
                           ('localGoals', '>=', 5)])

## 7. Particionado
Cuando los datos son muy grandes, en lugar de un único archivo se guardan en **una carpeta con subcarpetas por el valor de una columna**. Es lo que se conoce como *particionado*, y es la forma habitual de organizar un *data lake*.

In [ ]:
df.to_parquet('data/laliga_particionado', partition_cols = ['season'])

os.listdir('data/laliga_particionado')

Se ha creado una subcarpeta por temporada (`season=2005-06`, `season=2006-07`...), cada una con su archivo Parquet dentro.

In [ ]:
os.listdir('data/laliga_particionado/season=2010-11')

Para leer, basta con indicar la carpeta. `pandas` reconstruye la columna `season` a partir de los nombres de las subcarpetas.

In [ ]:
df_part = pd.read_parquet('data/laliga_particionado')
df_part.shape

Y si filtramos por la columna de partición, **directamente ni se abren las carpetas que no interesan**. En un data lake con años de datos, esto supone leer unos MB en lugar de cientos de GB.

In [ ]:
pd.read_parquet('data/laliga_particionado',
                filters = [('season', 'in', ['2016-17', '2017-18'])]).head()

## 8. ¿Cuándo usar cada formato?

| | CSV | Excel | JSON | pickle | Parquet |
|---|---|---|---|---|---|
| **Legible por humanos** | Sí | Sí (con Excel) | Sí | No | No |
| **Conserva tipos de datos** | No | Parcialmente | Parcialmente | Sí | Sí |
| **Tamaño en disco** | Grande | Medio | Grande | Medio | Muy pequeño |
| **Velocidad de lectura** | Lenta | Muy lenta | Lenta | Rápida | Muy rápida |
| **Leer solo algunas columnas** | No | No | No | No | Sí |
| **Entre lenguajes/herramientas** | Sí | Sí | Sí | Solo Python | Sí |
| **Datos anidados (no tabulares)** | No | No | Sí | Sí | Limitado |

**Usa Parquet cuando:**
* Tienes **datasets grandes** (cientos de MB o más).
* Guardas **datos intermedios ya limpios** en un pipeline (por ejemplo, entre la fase de limpieza y la de modelado de Machine Learning).
* Trabajas con herramientas de **Big Data o cloud**: Spark, Databricks, BigQuery, Athena, Snowflake, DuckDB, Polars...
* Quieres compartir datos con su **esquema y tipos** sin dar instrucciones de cómo leerlos.

**Mejor otro formato cuando:**
* Los datos son pequeños y alguien los va a **abrir o editar a mano**: CSV.
* El destinatario es **negocio**, que va a trabajar en una hoja de cálculo: Excel.
* Los datos son **jerárquicos/anidados** o vienen de una API: JSON.
* Necesitas guardar **objetos de Python** que no son tablas (un modelo entrenado, una clase propia...): pickle.

> **Ojo con pickle**: aunque conserva tipos y es rápido, solo funciona en Python, puede romperse entre versiones de librerías y **nunca debes cargar un pickle de una fuente que no sea de confianza**, porque puede ejecutar código. Para guardar tablas, Parquet es la opción segura y portable.

<table align="left">
 <tr><td width="80"><img src="./img/ejercicio.png" style="width:auto;height:auto"></td>
     <td style="text-align:left">
         <h3>Ejercicio Parquet</h3>

1. Lee `data/laliga.csv` y crea una columna `year` con el año de la fecha del partido (columna `date`). Guárdalo en formato Parquet en la carpeta `data/ejercicio_parquet`, con compresión `zstd` y particionado por `year`.
2. Lee de esa carpeta **solo** las columnas `localTeam` y `localGoals` de la temporada `2010-11`.
3. Compara cuánto ocupa el CSV original con lo que ocupa en total la carpeta Parquet (pista: recorre la carpeta con `os.walk()`).

 </td></tr>
</table>

**Limpieza**

Borramos los archivos generados en este notebook para no dejar basura en la carpeta `data`.

In [ ]:
import shutil

for archivo in os.listdir('data'):
    if archivo.endswith('.parquet') or archivo == 'laliga_tipos.csv':
        os.remove(os.path.join('data', archivo))

shutil.rmtree('data/laliga_particionado', ignore_errors = True)
shutil.rmtree('data/ejercicio_parquet', ignore_errors = True)